# Parameter sensitivity analysis

How sensitive is the along-track velocity RMSE to the drifter model
parameters $k_d$ (drogue drag coefficient) and $\tilde{m}_d$ (drogue
added mass)?

The steady-state drift velocity is an $\alpha$-weighted average of
surface and drogue-depth effective currents:

$$\alpha = \frac{\sqrt{k_b}}{\sqrt{k_b} + \sqrt{k_d}}$$

Since $\alpha$ depends only on $k_d$ (with $k_b$ fixed), we can sweep
$k_d$ cheaply by recomputing $\alpha$ and re-evaluating the prediction
at each observed drifter position -- no Parcels run needed.

The added mass $\tilde{m}_d$ affects only the transient adjustment
timescale (how fast the drifter reaches steady state) and does not
appear in the steady-state formula. A 2D grid search over both
parameters would require full ODE integration and is deferred to a
follow-up notebook.

## Imports

In [ ]:
import copernicusmarine as cm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

## Parameters

In [ ]:
CSV_PATH = "examples/baltic_drifters/data/drifters_clean.csv"
TIME_START = "2023-04-24"
TIME_END = "2023-05-10"
K_B = 12.0       # buoy drag coefficient [kg/m]
K_D_DEFAULT = 154.0  # default drogue drag coefficient [kg/m]
RESAMPLE_INTERVAL = "1h"
K_D_MIN = 10.0   # sweep range for k_d
K_D_MAX = 500.0
N_KD = 200       # number of k_d values in the sweep

## Load observed trajectories

Resample to hourly fixes and compute observed velocity from finite
differences. Same preprocessing as notebook 10.

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=["date_UTC"])

records = []
for drifter_id, grp in df.groupby("D_number"):
    grp = grp.set_index("date_UTC").sort_index()
    grp_h = grp.resample(RESAMPLE_INTERVAL).first().dropna(subset=["Latitude"])
    grp_h["D_number"] = drifter_id
    records.append(grp_h.reset_index())

df_h = pd.concat(records, ignore_index=True)

obs_records = []
for drifter_id, grp in df_h.groupby("D_number"):
    grp = grp.sort_values("date_UTC").reset_index(drop=True)
    dt = grp["date_UTC"].diff().dt.total_seconds()
    dlat = grp["Latitude"].diff()
    dlon = grp["Longitude"].diff()
    lat_rad = np.radians(grp["Latitude"])

    grp["u_obs"] = dlon * 1852 * 60 * np.cos(lat_rad) / dt
    grp["v_obs"] = dlat * 1852 * 60 / dt
    obs_records.append(grp.iloc[1:])

df_obs = pd.concat(obs_records, ignore_index=True)
print(f"{len(df_obs)} hourly fixes across {df_obs['D_number'].nunique()} drifters")

## Sample CMEMS effective currents along tracks

Interpolate Eulerian currents (surface and 3 m) and Stokes drift along
each drifter position and time. Same approach as notebook 10.

In [ ]:
TIME = slice(TIME_START, TIME_END)

lon_min, lon_max = df_obs["Longitude"].min() - 0.2, df_obs["Longitude"].max() + 0.2
lat_min, lat_max = df_obs["Latitude"].min() - 0.2, df_obs["Latitude"].max() + 0.2
LON = slice(lon_min, lon_max)
LAT = slice(lat_min, lat_max)

ds_phy = cm.open_dataset(
    dataset_id="cmems_mod_bal_phy_anfc_PT1H-i",
    service="arco-geo-series",
)[["uo", "vo"]].sel(
    longitude=LON, latitude=LAT, time=TIME, depth=slice(0, 5),
)

WAVE_VARS = [
    "VHM0_WW", "VTM01_WW", "VMDR_WW",
    "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",
    "VHM0_SW2", "VTM01_SW2", "VMDR_SW2",
]

ds_wav = cm.open_dataset(
    dataset_id="cmems_mod_bal_wav_anfc_PT1H-i",
    service="arco-geo-series",
).sel(longitude=LON, latitude=LAT, time=TIME)[WAVE_VARS]

print(f"Physics: {ds_phy.dims}")
print(f"Waves:   {ds_wav.dims}")

In [ ]:
times = xr.DataArray(pd.DatetimeIndex(df_obs["date_UTC"].values), dims="obs")
lons = xr.DataArray(df_obs["Longitude"].values, dims="obs")
lats = xr.DataArray(df_obs["Latitude"].values, dims="obs")

depth_levels = ds_phy.depth.values
z_surf = depth_levels[0]
z_3m = depth_levels[3]

euler_sampled = ds_phy.interp(
    longitude=lons, latitude=lats, time=times, method="linear",
).compute()

df_obs["uo_surf"] = euler_sampled["uo"].sel(depth=z_surf).values
df_obs["vo_surf"] = euler_sampled["vo"].sel(depth=z_surf).values
df_obs["uo_3m"] = euler_sampled["uo"].sel(depth=z_3m).values
df_obs["vo_3m"] = euler_sampled["vo"].sel(depth=z_3m).values

print(f"Eulerian sampled: {df_obs[['uo_surf','vo_surf']].notna().all(axis=1).sum()} valid of {len(df_obs)}")

### Stokes drift at surface and drogue depth

In [ ]:
g = 9.81

PARTITIONS = [
    ("VHM0_WW", "VTM01_WW", "VMDR_WW"),
    ("VHM0_SW1", "VTM01_SW1", "VMDR_SW1"),
    ("VHM0_SW2", "VTM01_SW2", "VMDR_SW2"),
]

wav_sampled = ds_wav.interp(
    longitude=lons, latitude=lats, time=times, method="linear",
).compute()

for label, z in [("0m", 0.0), ("3m", float(z_3m))]:
    u_st = np.zeros(len(df_obs))
    v_st = np.zeros(len(df_obs))

    for hs_var, t_var, dir_var in PARTITIONS:
        hs = wav_sampled[hs_var].values
        T = wav_sampled[t_var].values
        dir_from = wav_sampled[dir_var].values

        valid = (hs > 0.01) & np.isfinite(T) & (T > 0.1)
        A = np.where(valid, hs / 2, 0.0)
        sigma = np.where(valid, 2 * np.pi / np.where(valid, T, 1.0), 0.0)
        k = sigma**2 / g
        theta = np.deg2rad(270.0 - np.where(valid, dir_from, 0.0))
        decay = np.exp(-2 * k * z)
        mag = A**2 * sigma * k * decay

        u_st += mag * np.cos(theta)
        v_st += mag * np.sin(theta)

    df_obs[f"u_stokes_{label}"] = u_st
    df_obs[f"v_stokes_{label}"] = v_st

print(f"Stokes at surface: mean speed {np.sqrt(df_obs['u_stokes_0m']**2 + df_obs['v_stokes_0m']**2).mean():.4f} m/s")
print(f"Stokes at 3m:      mean speed {np.sqrt(df_obs['u_stokes_3m']**2 + df_obs['v_stokes_3m']**2).mean():.4f} m/s")

### Effective currents and valid subset

In [ ]:
df_obs["u_eff_surf"] = df_obs["uo_surf"] + df_obs["u_stokes_0m"]
df_obs["v_eff_surf"] = df_obs["vo_surf"] + df_obs["v_stokes_0m"]
df_obs["u_eff_3m"] = df_obs["uo_3m"] + df_obs["u_stokes_3m"]
df_obs["v_eff_3m"] = df_obs["vo_3m"] + df_obs["v_stokes_3m"]

df_valid = df_obs.dropna(subset=["uo_surf", "uo_3m"]).copy()
print(f"{len(df_valid)} valid fixes out of {len(df_obs)} ({len(df_obs) - len(df_valid)} dropped near coast)")

## 1D sensitivity: sweep $k_d$

For each $k_d$ value, compute $\alpha(k_d)$ and the alpha-weighted
velocity prediction, then evaluate RMSE against observed velocity.
This is pure numpy -- no Parcels runs needed.

In [ ]:
k_d_values = np.linspace(K_D_MIN, K_D_MAX, N_KD)
alpha_values = np.sqrt(K_B) / (np.sqrt(K_B) + np.sqrt(k_d_values))

# Precompute observed and effective current arrays for speed
u_obs = df_valid["u_obs"].values
v_obs = df_valid["v_obs"].values
u_eff_surf = df_valid["u_eff_surf"].values
v_eff_surf = df_valid["v_eff_surf"].values
u_eff_3m = df_valid["u_eff_3m"].values
v_eff_3m = df_valid["v_eff_3m"].values
drifter_ids_valid = df_valid["D_number"].values

# Overall RMSE for each k_d
rmse_all = np.empty(N_KD)
for i, alpha in enumerate(alpha_values):
    u_pred = (1 - alpha) * u_eff_3m + alpha * u_eff_surf
    v_pred = (1 - alpha) * v_eff_3m + alpha * v_eff_surf
    rmse_all[i] = np.sqrt(np.mean((u_obs - u_pred)**2 + (v_obs - v_pred)**2))

# Per-drifter RMSE
drifter_ids_list = sorted(df_valid["D_number"].unique())
rmse_per_drifter = np.empty((len(drifter_ids_list), N_KD))

for j, did in enumerate(drifter_ids_list):
    mask = drifter_ids_valid == did
    for i, alpha in enumerate(alpha_values):
        u_pred = (1 - alpha) * u_eff_3m[mask] + alpha * u_eff_surf[mask]
        v_pred = (1 - alpha) * v_eff_3m[mask] + alpha * v_eff_surf[mask]
        rmse_per_drifter[j, i] = np.sqrt(np.mean((u_obs[mask] - u_pred)**2 + (v_obs[mask] - v_pred)**2))

# Find optimal k_d (overall)
i_opt = np.argmin(rmse_all)
k_d_opt = k_d_values[i_opt]
alpha_opt = alpha_values[i_opt]
print(f"Optimal k_d = {k_d_opt:.1f} kg/m  (alpha = {alpha_opt:.3f}, RMSE = {rmse_all[i_opt]:.4f} m/s)")
print(f"Default k_d = {K_D_DEFAULT:.1f} kg/m  (alpha = {np.sqrt(K_B)/(np.sqrt(K_B)+np.sqrt(K_D_DEFAULT)):.3f}, RMSE = {rmse_all[np.argmin(np.abs(k_d_values - K_D_DEFAULT))]:.4f} m/s)")

### RMSE vs $k_d$: all drifters combined

In [ ]:
fig, ax1 = plt.subplots()

ax1.plot(k_d_values, rmse_all, "k-")
ax1.axvline(K_D_DEFAULT, color="tab:blue", ls="--", label=f"default $k_d$ = {K_D_DEFAULT:.0f}")
ax1.axvline(k_d_opt, color="tab:red", ls="--", label=f"optimal $k_d$ = {k_d_opt:.0f}")
ax1.set_xlabel("$k_d$ [kg/m]")
ax1.set_ylabel("Velocity RMSE [m/s]")
ax1.legend()

# Secondary axis: alpha
ax2 = ax1.twiny()
alpha_ticks = [0.05, 0.1, 0.2, 0.3, 0.5]
k_d_ticks = [K_B * ((1 - a) / a)**2 for a in alpha_ticks]
ax2.set_xlim(ax1.get_xlim())
ax2.set_xticks(k_d_ticks)
ax2.set_xticklabels([f"{a:.2f}" for a in alpha_ticks])
ax2.set_xlabel(r"$\alpha$")

plt.tight_layout()
plt.show()

### RMSE vs $k_d$: per drifter

In [ ]:
fig, ax = plt.subplots()

for j, did in enumerate(drifter_ids_list):
    ax.plot(k_d_values, rmse_per_drifter[j], label=f"D{did}", alpha=0.7)

ax.plot(k_d_values, rmse_all, "k-", linewidth=2, label="All")
ax.axvline(K_D_DEFAULT, color="0.5", ls="--", label=f"default $k_d$ = {K_D_DEFAULT:.0f}")
ax.set_xlabel("$k_d$ [kg/m]")
ax.set_ylabel("Velocity RMSE [m/s]")
ax.legend()
plt.tight_layout()
plt.show()

## Sensitivity in $\alpha$ space

Since $\alpha$ is the physical quantity that directly controls the
weighting, it is more natural to plot RMSE as a function of $\alpha$
rather than $k_d$.

In [ ]:
alpha_sweep = np.linspace(0.0, 1.0, 201)

rmse_alpha_all = np.empty(len(alpha_sweep))
for i, alpha in enumerate(alpha_sweep):
    u_pred = (1 - alpha) * u_eff_3m + alpha * u_eff_surf
    v_pred = (1 - alpha) * v_eff_3m + alpha * v_eff_surf
    rmse_alpha_all[i] = np.sqrt(np.mean((u_obs - u_pred)**2 + (v_obs - v_pred)**2))

rmse_alpha_per_drifter = np.empty((len(drifter_ids_list), len(alpha_sweep)))
for j, did in enumerate(drifter_ids_list):
    mask = drifter_ids_valid == did
    for i, alpha in enumerate(alpha_sweep):
        u_pred = (1 - alpha) * u_eff_3m[mask] + alpha * u_eff_surf[mask]
        v_pred = (1 - alpha) * v_eff_3m[mask] + alpha * v_eff_surf[mask]
        rmse_alpha_per_drifter[j, i] = np.sqrt(np.mean((u_obs[mask] - u_pred)**2 + (v_obs[mask] - v_pred)**2))

alpha_default = np.sqrt(K_B) / (np.sqrt(K_B) + np.sqrt(K_D_DEFAULT))
i_opt_alpha = np.argmin(rmse_alpha_all)
alpha_opt_direct = alpha_sweep[i_opt_alpha]
print(f"Optimal alpha = {alpha_opt_direct:.3f}  (RMSE = {rmse_alpha_all[i_opt_alpha]:.4f} m/s)")
print(f"Default alpha = {alpha_default:.3f}  (RMSE = {rmse_alpha_all[np.argmin(np.abs(alpha_sweep - alpha_default))]:.4f} m/s)")

In [ ]:
fig, ax = plt.subplots()

for j, did in enumerate(drifter_ids_list):
    ax.plot(alpha_sweep, rmse_alpha_per_drifter[j], label=f"D{did}", alpha=0.7)

ax.plot(alpha_sweep, rmse_alpha_all, "k-", linewidth=2, label="All")
ax.axvline(alpha_default, color="tab:blue", ls="--", label=f"default ({alpha_default:.3f})")
ax.axvline(alpha_opt_direct, color="tab:red", ls="--", label=f"optimal ({alpha_opt_direct:.3f})")

ax.set_xlabel(r"$\alpha$")
ax.set_ylabel("Velocity RMSE [m/s]")
ax.legend()
plt.tight_layout()
plt.show()

## Correlation sensitivity

The RMSE minimum may be shallow. Check whether the correlation between
predicted and observed velocity also peaks near the same $\alpha$.

In [ ]:
corr_alpha_all = np.empty(len(alpha_sweep))
for i, alpha in enumerate(alpha_sweep):
    u_pred = (1 - alpha) * u_eff_3m + alpha * u_eff_surf
    v_pred = (1 - alpha) * v_eff_3m + alpha * v_eff_surf
    r_u = np.corrcoef(u_obs, u_pred)[0, 1]
    r_v = np.corrcoef(v_obs, v_pred)[0, 1]
    corr_alpha_all[i] = (r_u + r_v) / 2

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

ax1.plot(alpha_sweep, rmse_alpha_all, "k-")
ax1.axvline(alpha_default, color="tab:blue", ls="--", label=f"default ({alpha_default:.3f})")
ax1.set_ylabel("Velocity RMSE [m/s]")
ax1.legend()

ax2.plot(alpha_sweep, corr_alpha_all, "k-")
ax2.axvline(alpha_default, color="tab:blue", ls="--")
ax2.set_xlabel(r"$\alpha$")
ax2.set_ylabel("Mean correlation (u, v)")

plt.tight_layout()
plt.show()

## Physical interpretation of $k_d$

Map the optimal $\alpha$ back to physical drogue parameters. Given
$k_d = \frac{1}{2}\rho\, C_{D,d}\, A_d$ and the Callies geometry
($w_d = h_d = 0.5$ m, $\rho = 1025$ kg/m$^3$), varying $k_d$ is
equivalent to varying the effective drag coefficient $C_{D,d}$ or the
drogue area $A_d$.

In [ ]:
rho = 1025.0   # kg/m^3
A_d = 0.25     # m^2 (w_d * h_d = 0.5 * 0.5)

# Implied C_D,d for the optimal k_d
C_D_d_opt = 2 * k_d_opt / (rho * A_d)
C_D_d_default = 2 * K_D_DEFAULT / (rho * A_d)

print(f"Default:  k_d = {K_D_DEFAULT:.0f} kg/m  ->  C_D,d = {C_D_d_default:.2f}")
print(f"Optimal:  k_d = {k_d_opt:.0f} kg/m  ->  C_D,d = {C_D_d_opt:.2f}")
print()

# Also express as equivalent drogue area at fixed C_D,d = 1.2
C_D_d_fixed = 1.2
A_d_opt = 2 * k_d_opt / (rho * C_D_d_fixed)
print(f"At C_D,d = {C_D_d_fixed}: optimal A_d = {A_d_opt:.3f} m^2  (default {A_d:.3f} m^2)")

## Optimal $k_d$ per drifter

Check whether the optimal $k_d$ is consistent across drifters or
whether some drifters prefer very different values.

In [ ]:
rows = []
for j, did in enumerate(drifter_ids_list):
    i_min = np.argmin(rmse_per_drifter[j])
    k_opt_j = k_d_values[i_min]
    a_opt_j = alpha_values[i_min]
    rows.append({
        "Drifter": did,
        "k_d_opt [kg/m]": k_opt_j,
        "alpha_opt": a_opt_j,
        "RMSE_opt [m/s]": rmse_per_drifter[j, i_min],
        "RMSE_default [m/s]": rmse_per_drifter[j, np.argmin(np.abs(k_d_values - K_D_DEFAULT))],
    })

# Overall
rows.append({
    "Drifter": "All",
    "k_d_opt [kg/m]": k_d_opt,
    "alpha_opt": alpha_opt,
    "RMSE_opt [m/s]": rmse_all[i_opt],
    "RMSE_default [m/s]": rmse_all[np.argmin(np.abs(k_d_values - K_D_DEFAULT))],
})

df_opt = pd.DataFrame(rows)
df_opt["RMSE_reduction [%]"] = (
    100 * (df_opt["RMSE_default [m/s]"] - df_opt["RMSE_opt [m/s]"]) / df_opt["RMSE_default [m/s]"]
).round(1)
df_opt

## 2D sensitivity: $k_d$ vs $\tilde{m}_d$ via ODE integration

The added mass $\tilde{m}_d$ does not affect steady-state $\alpha$ but
does affect the transient timescale. To see its effect on the
along-track RMSE, we integrate the full drifter ODE at each observed
position with varying ($k_d$, $\tilde{m}_d$) pairs using
`DroguedDrifter.get_final_drift_batch`.

We use a coarse 8x8 grid since each evaluation requires solving the ODE
system to steady state for all valid fixes.

In [ ]:
from drogued_drifters.drifter import DroguedDrifter

N_GRID = 8
k_d_2d = np.linspace(50, 300, N_GRID)
m_tilde_d_2d = np.linspace(20, 200, N_GRID)

# Effective currents in m/s at each valid fix
U_b = u_eff_surf.copy()
V_b = v_eff_surf.copy()
U_d = u_eff_3m.copy()
V_d = v_eff_3m.copy()

rmse_2d = np.empty((N_GRID, N_GRID))

for i, k_d in enumerate(k_d_2d):
    for j, m_td in enumerate(m_tilde_d_2d):
        dd = DroguedDrifter(k_d=k_d, m_tilde_d=m_td)
        xd_drift, yd_drift, _, _ = dd.get_final_drift_batch(
            U_b=U_b, V_b=V_b, U_d=U_d, V_d=V_d,
        )
        rmse_2d[i, j] = np.sqrt(np.mean((u_obs - xd_drift)**2 + (v_obs - yd_drift)**2))
    print(f"  k_d = {k_d:.0f} done")

print("2D grid search complete.")

### RMSE heatmap

In [ ]:
da_rmse = xr.DataArray(
    rmse_2d,
    dims=["k_d", "m_tilde_d"],
    coords={"k_d": k_d_2d, "m_tilde_d": m_tilde_d_2d},
    attrs={"long_name": "Velocity RMSE", "units": "m/s"},
)
da_rmse.coords["k_d"].attrs = {"long_name": "$k_d$", "units": "kg/m"}
da_rmse.coords["m_tilde_d"].attrs = {"long_name": r"$\tilde{m}_d$", "units": "kg"}

da_rmse.plot()
plt.plot(101, 154, "w*", markersize=15, markeredgecolor="k", label="default")
plt.legend()
plt.show()

# Print min
i_min, j_min = np.unravel_index(np.argmin(rmse_2d), rmse_2d.shape)
print(f"2D minimum: k_d = {k_d_2d[i_min]:.0f} kg/m, m_tilde_d = {m_tilde_d_2d[j_min]:.0f} kg, RMSE = {rmse_2d[i_min, j_min]:.4f} m/s")